## LeNet5 — Torch + LiteRT build pipeline

Trains a LeNet5 on MNIST, exports an int8 TFLite model via PT2E per-tensor quantization (analogous to the TensorFlow lenet notebook), using `litert-torch`.

Outputs

- `models/lenet5_torch_quantized.tflite` INT8 input/output, per-tensor; MicroFlow-compatible

Keep `epochs` small if you only need artifacts quickly.

In [7]:
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm

SEED = 3407
torch.manual_seed(SEED)

REPO_ROOT = Path.cwd().parent
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

OUT_TFLITE_QUANT = MODELS_DIR / "lenet5_torch_quantized.tflite"

DEVICE = "cpu"
print("Model output:", OUT_TFLITE_QUANT)
print("Device:", DEVICE)

Model output: /home/nathan/Documents/ariel-microflow-ml/models/lenet5_torch_quantized.tflite
Device: cpu


In [8]:
class LeNet5(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5),
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.Conv2d(6, 16, kernel_size=5),
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 120),
            nn.ReLU(),
            nn.Linear(120, 84),
            nn.ReLU(),
            nn.Linear(84, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


transform = transforms.Compose([transforms.ToTensor()])

train_ds = datasets.MNIST(root=str(REPO_ROOT / "dataset"), train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root=str(REPO_ROOT / "dataset"), train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

model = LeNet5().to(DEVICE)
print(model)

LeNet5(
  (features): Sequential(
    (0): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
    (1): ReLU()
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (3): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
    (4): ReLU()
    (5): AvgPool2d(kernel_size=2, stride=2, padding=0)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=256, out_features=120, bias=True)
    (2): ReLU()
    (3): Linear(in_features=120, out_features=84, bias=True)
    (4): ReLU()
    (5): Linear(in_features=84, out_features=10, bias=True)
  )
)


In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def evaluate(loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = model(xb).argmax(dim=1)
            correct += int((pred == yb).sum().item())
            total += yb.size(0)
    return correct / max(total, 1)


EPOCHS = 2
for epoch in range(EPOCHS):
    model.train()
    for xb, yb in tqdm(train_loader, desc=f"epoch {epoch + 1}"):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
    val_acc = evaluate(test_loader)
    print(f"epoch={epoch + 1} val_acc={val_acc:.4f}")

print("Test accuracy:", evaluate(test_loader))

epoch 1: 100%|██████████| 469/469 [00:45<00:00, 10.32it/s]


epoch=1 val_acc=0.9402


epoch 2: 100%|██████████| 469/469 [00:07<00:00, 60.11it/s]


epoch=2 val_acc=0.9713
Test accuracy: 0.9713


In [10]:
import litert_torch
from litert_torch.quantize import quant_config as qcfg
from litert_torch.quantize import pt2e_quantizer
from torchao.quantization.pt2e import quantize_pt2e

model.eval()
sample_input = torch.randn(1, 1, 28, 28, device=DEVICE)

exported = torch.export.export(model.cpu(), (sample_input.cpu(),))
m = exported.module()

quantizer = pt2e_quantizer.PT2EQuantizer().set_global(
    pt2e_quantizer.get_symmetric_quantization_config(is_per_channel=False)
)
m = quantize_pt2e.prepare_pt2e(m, quantizer)

calib_loader = DataLoader(test_ds, batch_size=1, shuffle=False)
with torch.no_grad():
    for i, (xb, _) in enumerate(calib_loader):
        if i >= 200:
            break
        m(xb.cpu())

m = quantize_pt2e.convert_pt2e(m, fold_quantize=False)
m.eval()

quant_cfg = qcfg.QuantConfig(pt2e_quantizer=quantizer)
edge_model = litert_torch.convert(m, (sample_input.cpu(),), quant_config=quant_cfg)

with torch.no_grad():
    torch_q_out = m(sample_input.cpu()).detach().cpu().numpy()
edge_output = edge_model(sample_input.cpu())

if np.allclose(torch_q_out, edge_output, atol=1e-2, rtol=1e-2):
    print("Quantized Torch vs LiteRT was within tolerance")
else:
    print("Quantized Torch vs LiteRT differed beyond tolerance")

edge_model.export(str(OUT_TFLITE_QUANT))
print("Wrote int8 per-tensor TFLite to", OUT_TFLITE_QUANT)


I0000 00:00:1776352694.450294   50889 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776352694.622900   50889 cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776352699.670139   50889 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.9.1+cu128).
/home/nath

NotImplementedError: 
        Calling train() or eval() is not supported for exported models.
        Please call `torchao.quantization.pt2e.move_exported_model_to_train(model)` (or eval) instead.

        If you cannot replace the calls to `model.train()` and `model.eval()`, you may override
        the behavior for these methods by calling `torchao.quantization.pt2e.allow_exported_model_train_eval(model)`,
        which does the above automatically for you. Note that this has limited effect on switching
        behavior between train and eval modes, and should be used only for special ops such as dropout
        and batchnorm.
        